# Comparativa: HMM hard con distintos K + 5 técnicas baseline - Weather
Test HMM hard con K={5, 8, 10, 12} (caches de train_hmm_multi_k_weather.ipynb).
Arquitectura reducida: d_model=16, n_heads=2, e_layers=1, d_ff=32.

In [ ]:
import os, subprocess, numpy as np

os.chdir('/home/jaime/TFG/RITMO')

# 5 técnicas baseline + HMM hard con 4 valores de K
TECHNIQUES = ['discretization', 'text_based', 'patching', 'decomposition', 'foundation']
HMM_K_VALUES = [5, 8, 10, 12]

BASE_CMD = [
    'python', '-u', 'run.py',
    '--task_name', 'plan_a',
    '--is_training', '1',
    '--root_path', './dataset/weather/',
    '--data_path', 'weather.csv',
    '--model_id', 'Weather_96_96',
    '--model', 'TransformerCommon',
    '--data', 'Weather',
    '--features', 'S',
    '--seq_len', '96',
    '--pred_len', '96',
    '--enc_in', '1',
    '--dec_in', '1',
    '--c_out', '1',
    '--d_model', '16',
    '--n_heads', '2',
    '--e_layers', '1',
    '--d_ff', '32',
    '--dropout', '0.1',
    '--batch_size', '32',
    '--learning_rate', '0.0001',
    '--train_epochs', '20',
    '--patience', '5',
    '--use_gpu', '0',
    '--itr', '1',
]

def setting_for(technique, K=None):
    if technique == 'hmm' and K is not None:
        des = f'Plan_A_hmm_K{K}'
    else:
        des = f'Plan_A_{technique}'
    return f'plan_a_Weather_96_96_TransformerCommon_Weather_ftS_sl96_ll48_pl96_dm16_nh2_el1_dl1_df32_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0'

def run_technique(technique, K=None):
    ckpt = f'./checkpoints/{setting_for(technique, K)}/checkpoint.pth'
    if os.path.exists(ckpt):
        label = f'{technique}' + (f' K={K}' if K else '')
        print(f'{label}: SKIP - checkpoint existe')
        return
    
    if technique == 'hmm' and K is not None:
        des = f'Plan_A_hmm_K{K}'
        cmd = BASE_CMD + ['--technique', 'hmm', '--hmm_k', str(K), '--des', des]
        label = f'hmm K={K}'
    else:
        des = f'Plan_A_{technique}'
        cmd = BASE_CMD + ['--technique', technique, '--des', des]
        label = technique
    
    print(f'{label}: ejecutando...')
    r = subprocess.run(cmd, capture_output=False)
    print(f'{label}: {"OK" if r.returncode == 0 else "ERROR"}')

print('Setup completo. Técnicas baseline + HMM K={5,8,10,12}')

In [ ]:
run_technique('discretization')

In [ ]:
run_technique('text_based')

In [ ]:
run_technique('patching')

In [ ]:
run_technique('decomposition')

In [ ]:
run_technique('foundation')

In [ ]:
run_technique('hmm', K=5)

In [ ]:
run_technique('hmm', K=8)

In [ ]:
run_technique('hmm', K=10)

In [ ]:
# Añadir celda para HMM K=12
run_technique('hmm', K=12)

In [ ]:
# Resumen de resultados: 5 técnicas baseline + HMM con 4 valores de K
print(f'{"Tecnica":>20s} | {"MSE":>10} | {"MAE":>10}')
print('-' * 48)

# Técnicas baseline
for tech in TECHNIQUES:
    setting = setting_for(tech)
    metrics_path = f'./results/{setting}/metrics.npy'
    if os.path.exists(metrics_path):
        mae, mse, rmse, mape, mspe = np.load(metrics_path)
        print(f'{tech:>20s} | {mse:10.6f} | {mae:10.6f}')
    else:
        print(f'{tech:>20s} | {"pendiente":>10} | {"pendiente":>10}')

# HMM con diferentes K
for K in HMM_K_VALUES:
    setting = setting_for('hmm', K)
    metrics_path = f'./results/{setting}/metrics.npy'
    label = f'hmm K={K}'
    if os.path.exists(metrics_path):
        mae, mse, rmse, mape, mspe = np.load(metrics_path)
        print(f'{label:>20s} | {mse:10.6f} | {mae:10.6f}')
    else:
        print(f'{label:>20s} | {"pendiente":>10} | {"pendiente":>10}')